# Your First llm-d Deployment

In this notebook, you will deploy the **llm-d optimized baseline** on a Kubernetes cluster.
This is the fastest way to get a production-ready LLM inference stack running with
intelligent routing, prefix caching, and load-aware scheduling.

By the end of this guide, you will have:
- A vLLM model server running Qwen3-32B
- The Endpoint Picker (EPP) router for intelligent request routing
- An Envoy gateway fronting the system with an OpenAI-compatible API
- A working curl command to send your first inference request

## Prerequisites

Before you begin, make sure you have:

- **A Kubernetes cluster** with at least one GPU node (NVIDIA A100 or H100 recommended)
- **kubectl** installed and configured to talk to your cluster
- **Helm** v3.x installed
- **kustomize** v5.x installed (or use `kubectl -k`)
- **Access to a model** hosted on Hugging Face (we will use Qwen/Qwen3-32B)

You can verify your tools are ready:

```bash
kubectl version --client
helm version
kustomize version
```

In [ ]:
# Clone the llm-d production stack repository
!git clone https://github.com/llm-d/llm-d-production-stack.git

# Set environment variables for this deployment
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["GUIDE_NAME"] = "optimized-baseline"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"
os.environ["HF_TOKEN"] = "<your-huggingface-token>"  # Replace with your token

print(f"Namespace: {os.environ['NAMESPACE']}")
print(f"Guide: {os.environ['GUIDE_NAME']}")
print(f"Model: {os.environ['MODEL_NAME']}")

## What Is the Optimized Baseline?

The **optimized baseline** is llm-d's recommended starting configuration. It combines two
key optimizations that work out of the box:

1. **Prefix-Cache-Aware Routing** - The router knows which model server replicas have
   which prompt prefixes cached in GPU memory. When a new request arrives, the router
   sends it to the replica most likely to have a cache hit, dramatically reducing
   time-to-first-token (TTFT).

2. **Load-Aware Routing** - The router monitors queue depths and KV-cache utilization
   across all replicas. It avoids sending requests to overloaded servers, preventing
   hotspots and improving tail latency.

These two signals are combined in a weighted scoring pipeline inside the
Endpoint Picker (EPP), giving you the best of both worlds.

In [ ]:
# Step 1: Create the namespace and deploy gateway provider prerequisites
# This installs the Gateway API CRDs and Envoy Gateway controller

!kubectl create namespace $NAMESPACE --dry-run=client -o yaml | kubectl apply -f -

# Install Gateway API CRDs
!kubectl apply -f https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.2.1/standard-install.yaml

# Install Envoy Gateway
!helm install eg oci://docker.io/envoyproxy/gateway-helm \
    --version v1.3.0 \
    -n envoy-gateway-system \
    --create-namespace

In [ ]:
# Step 2: Deploy the model server (vLLM serving Qwen3-32B)
# This creates a Deployment with GPU resource requests and the vLLM container

!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/model-server/ \
    -n $NAMESPACE

print("\nModel server deployment submitted.")
print("Note: The model server will take several minutes to start as it downloads")
print("the model weights and loads them into GPU memory.")

In [ ]:
# Step 3: Deploy the router (EPP + Envoy)
# The EPP handles intelligent routing decisions;
# Envoy proxies traffic and exposes the OpenAI-compatible API

!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/router/ \
    -n $NAMESPACE

print("\nRouter deployment submitted.")

## What Just Got Deployed?

You now have the following running in your cluster:

```
Client Request
     |
     v
+-------------+
| Envoy       |  <-- OpenAI-compatible API (port 8080)
| Gateway     |
+------+------+
       |
       v
+-------------+
| EPP Router  |  <-- Scores replicas by prefix-cache match + load
+------+------+
       |
  +----+----+
  |         |
  v         v
+----+  +----+
|vLLM|  |vLLM|  <-- Model server replicas with Qwen3-32B
+----+  +----+
```

- **Envoy Gateway**: Accepts incoming HTTP requests on port 8080 in the standard
  OpenAI chat completions format.
- **EPP (Endpoint Picker)**: An ext-proc filter that intercepts each request and
  picks the best backend replica based on prefix-cache affinity and load.
- **vLLM Model Servers**: Run the actual model inference with automatic prefix caching
  enabled.

In [ ]:
# Check pod status - wait for everything to be Running
!kubectl get pods -n $NAMESPACE -o wide

print("\n--- Waiting for all pods to be ready (this may take a few minutes) ---")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s

print("\nAll pods are ready!")

In [ ]:
# Send a test request through the gateway
# First, get the gateway endpoint

!export GATEWAY_IP=$(kubectl get svc -n $NAMESPACE envoy-gateway -o jsonpath='{.status.loadBalancer.ingress[0].ip}') && \
 echo "Gateway IP: $GATEWAY_IP" && \
 curl -s http://$GATEWAY_IP:8080/v1/chat/completions \
   -H "Content-Type: application/json" \
   -d '{
     "model": "Qwen/Qwen3-32B",
     "messages": [
       {"role": "user", "content": "What is Kubernetes in one sentence?"}
     ],
     "max_tokens": 100
   }' | python3 -m json.tool

## Congratulations!

You have deployed your first llm-d inference stack. Your deployment includes
intelligent prefix-cache-aware routing and load balancing out of the box.

### What to Try Next

llm-d provides several **well-lit paths** for more advanced configurations:

- **Sending Requests** - Learn the OpenAI-compatible API in depth, including
  streaming, multi-turn conversations, and observing cache hit rates.
- **Prefill/Decode Disaggregation** - Separate prefill and decode phases onto
  different GPU pools for maximum throughput at scale.
- **Flow Control** - Add priority bands and multi-tenant fairness to your
  inference gateway.
- **Autoscaling** - Configure SLO-aware horizontal pod autoscaling driven by
  inference-specific metrics like queue depth.

Check out the other notebooks in this series to explore these paths.